# 09 --- Advanced Features

Templates, schema discovery, manual pipelines, storage, eval harness, and MCP server.

## Beyond the standard pipeline

The previous notebooks covered DocFlow's core workflow: parse, anonymize, extract, validate, review, monitor. This notebook covers the tools that sit around that core — features that save time, enable new workflows, or address production requirements.

### Templates as encoded domain knowledge

Building a good Pydantic schema requires understanding the document type: what fields exist, which are required, what types they have, and how to describe them so the LLM extracts them correctly. DocFlow ships with **built-in templates** for common document types (invoices, contracts, receipts) that encode this knowledge. Loading a template gives you a production-ready schema without writing any Python.

### Schema discovery as bootstrapping

When you encounter an unfamiliar document type, you can ask the LLM to *read the document and suggest a schema*. `discover_schema()` returns a list of discovered fields with names, types, and descriptions, plus a ready-to-use Pydantic model. This is a bootstrapping tool — the discovered schema is a starting point that you refine, not a finished product.

### Manual pipelines for custom workflows

`DocumentPipeline` handles the common case, but sometimes you need custom step ordering, conditional logic, or steps that don't exist in the standard pipeline. The low-level `Pipeline` class lets you compose individual step objects (`Ingest`, `Parse`, `Anonymize`, `Extract`, `Validate`, `Review`, `Store`) in any order, with any configuration. This is how you build workflows that don't fit the standard pattern.

### Storage for audit and compliance

Production document processing often requires keeping the original file, the parsed content, the extraction result, and the full trace of what happened. `LocalDocumentStore` handles this — it persists everything to disk in a queryable structure, with partial-save on failure so you never lose work mid-pipeline.

### Eval harness for accuracy measurement

How do you know if your pipeline is actually getting the right answers? The `EvalHarness` compares extraction results against ground truth (human-corrected, approved results) and reports overall accuracy, per-field accuracy, and hallucination rates. This is how you measure whether a model change, schema tweak, or parser swap actually improved things.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

## Built-in templates

DocFlow ships with YAML templates for common document types. Load one and inspect its fields.

In [ ]:
from docflow.templates import load_template, list_templates

print(f"Available templates: {list_templates()}")

InvoiceTemplate = load_template("invoice")
print(f"\nInvoice template fields:")
for name, field in InvoiceTemplate.model_fields.items():
    req = "required" if field.is_required() else "optional"
    print(f"  {name:<22} ({req})")

## Extract using a template

Pass a loaded template as the schema to `extract()`.

In [ ]:
from docflow import extract

result = extract(PDF_PATH, schema=InvoiceTemplate, model="gemini/gemini-2.5-flash")

print(f"Extracted using template:")
for k, v in result.data.items():
    print(f"  {k:<22}: {v}")

## Schema discovery

Auto-generate a schema from a document. The LLM reads the content and suggests fields.

In [ ]:
from docflow import discover_schema

discovery = discover_schema(PDF_PATH, model="gemini/gemini-2.5-flash")

print(f"Document type: {discovery.document_type}")
print(f"\nDiscovered fields:")
for f in discovery.fields:
    print(f"  {f.name:<22} ({f.type}): {f.description}")

## Use the discovered schema

The discovery result includes a dynamically-built Pydantic model ready for extraction.

In [ ]:
AutoSchema = discovery.schema_class

result_auto = extract(PDF_PATH, schema=AutoSchema, model="gemini/gemini-2.5-flash")

print(f"Extraction with discovered schema:")
for k, v in result_auto.data.items():
    print(f"  {k:<22}: {v}")

## Manual Pipeline

For full control over every step, build a `Pipeline` from individual step objects.

In [ ]:
from docflow.workflow.pipeline import Pipeline
from docflow.workflow.steps import Ingest, Parse, Extract
from docflow.parsing.pymupdf import PyMuPDFParser
from docflow.extraction.llm.litellm_adapter import LiteLLMAdapter
from pydantic import BaseModel, Field


class SimpleInvoice(BaseModel):
    supplier: str = Field(description="Supplier name")
    total: float = Field(description="Total amount")
    date: str = Field(description="Invoice date")


llm = LiteLLMAdapter(model="gemini/gemini-2.5-flash")

manual_pipeline = Pipeline([
    Ingest(path=PDF_PATH),
    Parse(parser=PyMuPDFParser()),
    Extract(schema=SimpleInvoice, llm=llm),
])

pipe_result = manual_pipeline.run_sync()

print(f"Success: {pipe_result.success}")
print(f"Steps executed: {len(pipe_result.trace.events)}")
ext = pipe_result.state.extraction_result
print(f"Extracted: {ext.data}")

## Storage

Persist documents and extraction results locally with `LocalDocumentStore`.

In [ ]:
import tempfile
from docflow import DocumentPipeline
from docflow.storage.local import LocalDocumentStore

store_path = os.path.join(tempfile.mkdtemp(), "docflow_store")
store = LocalDocumentStore(store_path)

storage_pipeline = DocumentPipeline(
    parser="pymupdf",
    model="gemini/gemini-2.5-flash",
    storage=store,
)

result_stored = storage_pipeline.run_sync(PDF_PATH, schema=SimpleInvoice)
print(f"Result stored. Document ID: {result_stored.document_id}")

In [ ]:
import glob

saved_files = glob.glob(os.path.join(store_path, "**", "*"), recursive=True)
print(f"Saved files in store:")
for f in saved_files:
    rel = os.path.relpath(f, store_path)
    print(f"  {rel}")

## Eval harness

Measure extraction accuracy against ground truth. Corrections on an approved result become your gold set.

In [ ]:
from docflow.eval import EvalHarness
import copy

# Create ground truth from a corrected result
gold = copy.deepcopy(result)
gold.correct_field("supplier_name", "Meridian Dynamics Ltd", corrected_by="analyst")
gold.approve(approved_by="analyst")

harness = EvalHarness()
harness.add_ground_truth(gold)

# Compare a new prediction against ground truth
report_eval = harness.compare_results(predicted=[result])

print(f"Overall accuracy:    {report_eval.overall_accuracy:.2%}")
print(f"Hallucination rate:  {report_eval.hallucination_rate:.2%}")
print(f"\nPer-field accuracy:")
for fname, acc in report_eval.field_accuracy.items():
    print(f"  {fname:<22}: {acc:.2%}")

## MCP server

DocFlow can run as an MCP server exposing 14 tools for AI agents:

```
docflow-mcp
```

Tools: `extract_document`, `extract_with_vision`, `discover_schema`, `compare_documents`, `process_batch`, `list_templates`, `show_template`, `search_in_document`, `get_pending_reviews`, `get_extraction_result`, `correct_field`, `approve_document`, `reject_document`, `screenshot_document`.

## CLI reference

Quick reference of common CLI commands:

```
docflow extract file.pdf --schema invoice --output result.json
docflow extract-folder ./invoices --schema invoice --output results.csv
docflow run workflow.yaml invoice.pdf --output result.json
docflow screenshot file.pdf -o ./pages --dpi 200
docflow templates list
docflow templates show invoice
```